# 🛡️ Phishing Email Detection — End-to-End ML Workflow in Snowflake

**Course Assignment** | Snowflake ML Pipeline

---

## 📋 Table of Contents
1. [Introduction & Objective](#1-introduction)
2. [Snowflake Setup & Connection](#2-setup)
3. [Data Loading & Exploration (EDA)](#3-eda)
4. [Feature Engineering](#4-features)
5. [Model Training (Multiple Algorithms)](#5-training)
6. [Model Evaluation & Comparison](#6-evaluation)
7. [Feature Importance](#7-importance)
8. [Model Deployment (UDF)](#8-deployment)
9. [Real-Time Inference](#9-inference)
10. [Summary & Conclusion](#10-summary)

---

## 1. Introduction & Objective

**Problem Statement:** Phishing emails are one of the most common cyber-attack vectors. They trick users into revealing sensitive information by impersonating legitimate organizations.

**Objective:** Build an end-to-end ML pipeline **entirely within Snowflake** using **Snowpark Python** to:
- Ingest and store email data in Snowflake tables
- Perform Exploratory Data Analysis (EDA) using SQL and Python
- Engineer meaningful features from email metadata
- Train and compare multiple classification models
- Deploy the best model as a Snowflake UDF for real-time scoring

**Tech Stack:**
| Component | Technology |
|-----------|------------|
| Data Platform | Snowflake (Cloud Data Warehouse) |
| ML Framework | Snowpark Python + Scikit-learn |
| Model Deployment | Snowflake UDF + Stage |
| Inference | Snowflake SQL via UDF |

---
## 2. Snowflake Setup & Connection

In [ ]:
# Install required packages (run once)
# !pip install snowflake-snowpark-python scikit-learn pandas joblib matplotlib seaborn

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark import Session
from snowflake.snowpark.functions import col

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
import joblib, os

print('✅ All packages imported successfully!')

In [ ]:
# ⚠️ REPLACE WITH YOUR SNOWFLAKE TRIAL ACCOUNT CREDENTIALS
# Find your account identifier in Snowsight: Admin > Accounts
# Format: <orgname>-<accountname> (e.g., 'myorg-myaccount')

CONNECTION_PARAMETERS = {
    'account':   'YOUR_ACCOUNT_IDENTIFIER',  # e.g. 'abc12345.us-east-1'
    'user':      'YOUR_USERNAME',
    'password':  'YOUR_PASSWORD',
    'role':      'SYSADMIN',
    'warehouse': 'PHISHING_ML_WH',
    'database':  'PHISHING_DB',
    'schema':    'PHISHING_SCHEMA'
}

session = Session.builder.configs(CONNECTION_PARAMETERS).create()

print('✅ Connected to Snowflake!')
print(f'   Account  : {session.get_current_account()}')
print(f'   User     : {session.get_current_user()}')
print(f'   Warehouse: {session.get_current_warehouse()}')
print(f'   Database : {session.get_current_database()}')
print(f'   Schema   : {session.get_current_schema()}')

---
## 3. Data Loading & Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

Our dataset contains **50 emails** (25 phishing, 25 legitimate) with **20 engineered features** extracted from:
- **Header metadata** (SPF, DKIM, sender domain)
- **URL characteristics** (count, length, IP-based)
- **Content signals** (urgency words, special chars, word count)
- **Authentication** (SPF pass, DKIM pass, reply-to mismatch)

In [ ]:
# Load data from Snowflake
snow_df = session.table('ENGINEERED_FEATURES')
df = snow_df.to_pandas()
df.columns = df.columns.str.lower()

print(f'Dataset shape: {df.shape}')
print(f'\nColumn names:')
print(df.columns.tolist())
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Class Distribution
label_counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Phishing Email Dataset - EDA', fontsize=16, fontweight='bold')

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(
    label_counts,
    labels=['Legitimate', 'Phishing'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05)
)
axes[0].set_title('Class Distribution')

# Bar chart - feature averages by class
features_to_plot = ['has_url', 'has_ip_url', 'has_urgent_words',
                     'sender_name_mismatch', 'auth_both_failed', 'high_risk_combo']
class_means = df.groupby('label')[features_to_plot].mean()

x = np.arange(len(features_to_plot))
width = 0.35
axes[1].bar(x - width/2, class_means.loc[0], width, label='Legitimate', color='#2ecc71', alpha=0.8)
axes[1].bar(x + width/2, class_means.loc[1], width, label='Phishing',   color='#e74c3c', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    ['Has URL', 'IP URL', 'Urgent Words', 'Name Mismatch', 'Auth Failed', 'High Risk'],
    rotation=20, ha='right'
)
axes[1].set_title('Key Feature Comparison by Class')
axes[1].set_ylabel('Average Value')
axes[1].legend()
axes[1].set_ylim(0, 1.2)

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution:', dict(zip(['Legitimate', 'Phishing'], label_counts.values)))

In [ ]:
# Correlation Heatmap
FEATURE_COLS = [
    'has_url', 'url_count', 'has_ip_url', 'url_length',
    'has_urgent_words', 'html_tags_count', 'num_special_chars',
    'word_count', 'subject_length', 'sender_name_mismatch',
    'has_attachment', 'reply_to_different', 'spf_pass', 'dkim_pass',
    'auth_both_failed', 'url_density', 'special_char_ratio',
    'high_risk_combo', 'has_long_url', 'short_body', 'label'
]

plt.figure(figsize=(16, 14))
corr_matrix = df[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical Summary by class
print('=== Statistical Summary by Email Class ===')
numeric_feats = ['url_count', 'url_length', 'word_count', 'html_tags_count',
                 'num_special_chars', 'subject_length']
summary = df.groupby('label')[numeric_feats].agg(['mean', 'std', 'min', 'max']).round(2)
summary.index = ['Legitimate (0)', 'Phishing (1)']
print(summary)

---
## 4. Feature Engineering

Beyond the raw features, we engineered **5 additional derived features** that combine multiple signals to create stronger predictors:

| Derived Feature | Formula | Intuition |
|---|---|---|
| `auth_both_failed` | SPF=0 AND DKIM=0 | Strong phishing signal |
| `url_density` | url_count / word_count | High URL density is suspicious |
| `special_char_ratio` | special_chars / word_count | Obfuscation signal |
| `high_risk_combo` | urgent + mismatch + auth_fail | Multi-factor risk |
| `has_long_url` | url_length > 75 | Redirecting URLs |
| `short_body` | word_count < 20 | Sparse phishing emails |

In [ ]:
# Define feature matrix and target
FEATURES = [
    'has_url', 'url_count', 'has_ip_url', 'url_length',
    'has_urgent_words', 'html_tags_count', 'num_special_chars',
    'word_count', 'subject_length', 'sender_name_mismatch',
    'has_attachment', 'reply_to_different', 'spf_pass', 'dkim_pass',
    'auth_both_failed', 'url_density', 'special_char_ratio',
    'high_risk_combo', 'has_long_url', 'short_body'
]

X = df[FEATURES]
y = df['label']

# Train/test split (matching SQL MOD logic)
train_mask = df['email_id'].apply(lambda x: x % 5 != 0)
test_mask  = df['email_id'].apply(lambda x: x % 5 == 0)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'✅ Feature matrix: {X.shape}')
print(f'✅ Training set  : {X_train.shape[0]} samples')
print(f'✅ Test set      : {X_test.shape[0]} samples')
print(f'\nFeature correlation with label (top 10):')
corr_with_label = df[FEATURES + ['label']].corr()['label'].drop('label')
print(corr_with_label.abs().sort_values(ascending=False).head(10))

---
## 5. Model Training

We train **4 classification algorithms** and compare their performance:
1. **Logistic Regression** — Linear baseline model
2. **Decision Tree** — Interpretable rule-based model
3. **Random Forest** — Ensemble of decision trees (reduces variance)
4. **Gradient Boosting** — Sequential boosting for high accuracy

In [ ]:
from sklearn.model_selection import cross_val_score

# Define all models
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(random_state=42, max_depth=6))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8))
    ]),
    'Gradient Boosting': Pipeline([
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42, learning_rate=0.1))
    ])
}

results = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_proba)
    cv   = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    
    results[name] = {
        'pipeline': pipeline, 'y_pred': y_pred, 'y_proba': y_proba,
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'auc_roc': auc, 'cv_mean': cv.mean(), 'cv_std': cv.std()
    }
    print(f'{name:25s} | Acc: {acc:.3f} | F1: {f1:.3f} | AUC: {auc:.3f} | CV-F1: {cv.mean():.3f}±{cv.std():.3f}')

print('\n✅ All models trained!')

---
## 6. Model Evaluation & Comparison

In [ ]:
# Model Comparison Chart
metrics_df = pd.DataFrame({
    name: {
        'Accuracy':  res['accuracy'],
        'Precision': res['precision'],
        'Recall':    res['recall'],
        'F1 Score':  res['f1'],
        'AUC-ROC':   res['auc_roc']
    }
    for name, res in results.items()
}).T

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

# Bar chart comparison
metrics_df.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='black', linewidth=0.5)
axes[0].set_title('All Metrics by Model')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.15)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='x', rotation=15)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.2f}', 
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=7)

# ROC Curves
colors_roc = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={res['auc_roc']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Best Model - Confusion Matrix
best_model_name = max(results, key=lambda k: results[k]['f1'])
best_result = results[best_model_name]
best_pipeline = best_result['pipeline']

print(f'🏆 Best Model: {best_model_name}')
print(f'   F1 Score : {best_result["f1"]:.4f}')
print(f'   AUC-ROC  : {best_result["auc_roc"]:.4f}\n')

# Confusion matrix
cm = confusion_matrix(y_test, best_result['y_pred'])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Legitimate', 'Phishing'],
    yticklabels=['Legitimate', 'Phishing'],
    linewidths=1, linecolor='white',
    annot_kws={'size': 18, 'weight': 'bold'}
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, best_result['y_pred'],
                             target_names=['Legitimate', 'Phishing']))

---
## 7. Feature Importance

In [ ]:
# Feature Importance from Random Forest
rf_clf = results['Random Forest']['pipeline'].named_steps['clf']
importances = pd.Series(rf_clf.feature_importances_, index=FEATURES)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = ['#e74c3c' if imp > importances.mean() else '#3498db'
              for imp in importances_sorted]
bars = ax.barh(importances_sorted.index, importances_sorted.values,
               color=colors_fi, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, importances_sorted.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.axvline(x=importances.mean(), color='gray', linestyle='--',
            alpha=0.7, label=f'Mean importance ({importances.mean():.4f})')
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Feature Importance — Random Forest\n(Red = Above Average)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim(0, importances_sorted.max() * 1.15)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 Most Important Features:')
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f'  {feat:<30} {imp:.4f}')

---
## 8. Model Deployment to Snowflake Stage

In [ ]:
# Save best model and upload to Snowflake stage
model_file = 'phishing_detector_model.pkl'
joblib.dump(best_pipeline, model_file)
print(f'✅ Model saved locally: {model_file}')

# Create model stage
session.sql('CREATE STAGE IF NOT EXISTS PHISHING_MODEL_STAGE').collect()

# Upload to stage
session.file.put(model_file, '@PHISHING_MODEL_STAGE', overwrite=True)
print(f'✅ Model uploaded to @PHISHING_MODEL_STAGE')

# Write metrics to Snowflake
metrics_rows = []
for name, res in results.items():
    metrics_rows.append({
        'MODEL_NAME': name, 'ACCURACY': res['accuracy'],
        'PRECISION_SCORE': res['precision'], 'RECALL_SCORE': res['recall'],
        'F1_SCORE': res['f1'], 'AUC_ROC': res['auc_roc'],
        'TRAINED_AT': pd.Timestamp.now()
    })

session.sql('TRUNCATE TABLE IF EXISTS MODEL_METRICS').collect()
session.create_dataframe(pd.DataFrame(metrics_rows)).write.mode('append').save_as_table('MODEL_METRICS')
print('✅ Metrics written to MODEL_METRICS table')

# Verify in Snowflake
print('\nSnowflake Model Metrics:')
session.sql('SELECT MODEL_NAME, ROUND(ACCURACY*100,2) AS ACC_PCT, ROUND(F1_SCORE*100,2) AS F1_PCT FROM MODEL_METRICS ORDER BY F1_SCORE DESC').show()

---
## 9. Real-Time Inference via Snowflake UDF

In [ ]:
# Register the prediction UDF in Snowflake
@session.udf.register(
    name='predict_phishing',
    is_permanent=True,
    stage_location='@PHISHING_MODEL_STAGE',
    packages=['scikit-learn', 'joblib', 'pandas'],
    replace=True,
    return_type='float',
    input_types=['float'] * 20
)
def predict_phishing(*feature_values):
    import joblib, pandas as pd, sys, os
    model_path = os.path.join(
        sys._xoptions.get('snowflake_import_directory', ''),
        'phishing_detector_model.pkl'
    )
    model = joblib.load(model_path)
    cols = [
        'has_url','url_count','has_ip_url','url_length','has_urgent_words',
        'html_tags_count','num_special_chars','word_count','subject_length',
        'sender_name_mismatch','has_attachment','reply_to_different',
        'spf_pass','dkim_pass','auth_both_failed','url_density',
        'special_char_ratio','high_risk_combo','has_long_url','short_body'
    ]
    df = pd.DataFrame([list(feature_values)], columns=cols)
    return float(model.predict_proba(df)[0][1])

print('✅ UDF predict_phishing() registered in Snowflake!')

# Test real-time scoring on all emails
print('\n🔍 Real-time scoring via SQL UDF:')
session.sql("""
    SELECT EMAIL_ID, LABEL AS ACTUAL,
           ROUND(predict_phishing(
               HAS_URL, URL_COUNT, HAS_IP_URL, URL_LENGTH, HAS_URGENT_WORDS,
               HTML_TAGS_COUNT, NUM_SPECIAL_CHARS, WORD_COUNT, SUBJECT_LENGTH,
               SENDER_NAME_MISMATCH, HAS_ATTACHMENT, REPLY_TO_DIFFERENT,
               SPF_PASS, DKIM_PASS, AUTH_BOTH_FAILED, URL_DENSITY,
               SPECIAL_CHAR_RATIO, HIGH_RISK_COMBO, HAS_LONG_URL, SHORT_BODY
           ), 4) AS PHISHING_PROB
    FROM ENGINEERED_FEATURES
    ORDER BY PHISHING_PROB DESC
    LIMIT 10
""").show()

---
## 10. Summary & Conclusion

### ✅ Pipeline Completion Summary

| Stage | Status | Details |
|---|---|---|
| Data Ingestion | ✅ Done | 50 emails loaded to Snowflake |
| EDA | ✅ Done | Statistical analysis, correlation matrix |
| Feature Engineering | ✅ Done | 20 features (15 raw + 5 derived) |
| Model Training | ✅ Done | 4 algorithms compared |
| Model Evaluation | ✅ Done | Accuracy, F1, AUC-ROC, Confusion Matrix |
| Model Deployment | ✅ Done | Saved to @PHISHING_MODEL_STAGE |
| Real-Time Inference | ✅ Done | Registered as Snowflake UDF |

### 🔑 Key Findings

**Top phishing indicators (by feature importance):**
1. `auth_both_failed` — Both SPF and DKIM failed
2. `has_urgent_words` — Contains urgency language
3. `sender_name_mismatch` — Display name ≠ actual domain
4. `high_risk_combo` — Multi-factor risk combination
5. `spf_pass` / `dkim_pass` — Authentication signals

### 💡 Snowflake Advantages for ML
- **No data movement**: ML runs where data lives
- **Scalable**: Warehouse size controls compute power
- **Governance**: All data, models, and predictions in one platform
- **Democratized inference**: SQL UDF means any user can score emails

In [ ]:
# Final Summary
best_model_name = max(results, key=lambda k: results[k]['f1'])
best_result = results[best_model_name]

print('=' * 55)
print('  PHISHING DETECTION PIPELINE — FINAL RESULTS')
print('=' * 55)
print(f'Dataset            : 50 emails (25 phishing, 25 legit)')
print(f'Features           : 20 (15 raw + 5 derived)')
print(f'Models Trained     : 4 algorithms')
print(f'Best Model         : {best_model_name}')
print(f'  Accuracy         : {best_result["accuracy"]*100:.2f}%')
print(f'  Precision        : {best_result["precision"]*100:.2f}%')
print(f'  Recall           : {best_result["recall"]*100:.2f}%')
print(f'  F1 Score         : {best_result["f1"]*100:.2f}%')
print(f'  AUC-ROC          : {best_result["auc_roc"]:.4f}')
print(f'Deployment         : @PHISHING_MODEL_STAGE + UDF')
print('=' * 55)

session.close()
print('\n✅ Snowflake session closed.')